In [0]:
df_place = spark.read.format("delta").load(silver_path + "danger_place/")
df_tourist = spark.read.format("delta").load(silver_path + "tourist/")
df_agency = spark.read.format("delta").load(silver_path + "agency/")
df_trip = spark.read.format("delta").load(silver_path + "trip/")
df_alert = spark.read.format("delta").load(silver_path + "geofence_alert/")

metrics for gold

In [0]:
from pyspark.sql.functions import *

gold_danger_zone_metrics = df_alert.join(
    df_place,
    df_alert["PlaceId"] == df_place["DangerPlaceId"],
    "left"
).groupBy(
    df_place["DangerPlaceId"],
    df_place["Name"],
    df_place["Severity"]
).agg(
    count("AlertId").alias("TotalAlerts"),
    countDistinct("TouristId").alias("UniqueTourists"),
    avg("DistanceMeters").alias("AvgDistanceMeters"),
    sum(when(col("IsResolved") == False, 1).otherwise(0)).alias("UnresolvedAlerts")
).orderBy(col("TotalAlerts").desc())
display(gold_danger_zone_metrics)

(
    gold_danger_zone_metrics.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(gold_path + "gold_danger_zone_metrics/")
)

In [0]:
gold_tourist_risk_score = df_alert.groupBy("TouristId").agg(
    count("AlertId").alias("AlertFrequency"),
    sum(
        when(upper(col("Severity")) == "HIGH", 10)
        .when(upper(col("Severity")) == "MEDIUM", 5)
        .otherwise(2)
    ).alias("RiskScore"),
    sum(when(col("IsResolved") == False, 1).otherwise(0)).alias("UnresolvedAlerts")
).withColumn(
    "RiskCategory",
    when(col("RiskScore") >= 30, "High Risk")
    .when(col("RiskScore") >= 15, "Medium Risk")
    .otherwise("Low Risk")
)
(
    gold_tourist_risk_score.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(gold_path + "gold_tourist_risk_score/")
)

In [0]:
gold_agency_safety_metrics = df_trip.join(
    df_agency,
    "AgencyId",
    "left"
).join(
    df_alert,
    "TouristId",
    "left"
).groupBy(
    "AgencyId",
    "AgencyName"
).agg(
    countDistinct("TouristId").alias("TotalTourists"),
    count("AlertId").alias("TotalIncidents"),
    sum(when(col("IsResolved") == False, 1).otherwise(0)).alias("UnresolvedIncidents")
).withColumn(
    "SafetyScore",
    greatest(lit(0), lit(100) - (col("TotalIncidents") * 5) - (col("UnresolvedIncidents") * 10))
)
(
    gold_agency_safety_metrics.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(gold_path + "gold_agency_safety_metrics/")
)

In [0]:
gold_unresolved_alerts_summary = df_alert.filter(
    col("IsResolved") == False
).groupBy("PlaceId").agg(
    count("AlertId").alias("UnresolvedAlertCount")
)
(
    gold_unresolved_alerts_summary.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(gold_path + "gold_unresolved_alerts_summary/")
)

In [0]:
gold_daily_alert_trend = df_alert.withColumn(
    "AlertDate",
    to_date(col("CreatedAt"))
).groupBy("AlertDate").agg(
    count("AlertId").alias("TotalAlerts"),
    countDistinct("TouristId").alias("UniqueTourists")
).orderBy("AlertDate")
(
    gold_daily_alert_trend.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(gold_path+"/daily_alert_trend")
)

In [0]:
gold_severity_distribution = df_alert.groupBy("Severity").agg(
    count("AlertId").alias("TotalAlerts")
)
(
    gold_severity_distribution.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(gold_path+"/severity_distribution")
)